# 02 · Lấy mẫu, ghép và lọc playlist (ticket #5)

**Input** (Add Input): `himanshuwagh/spotify-million`, `krishsharma0413/2-million-songs-from-mpd-with-audio-features`.

**Output** (`/kaggle/working/processed/`): `sessions_raw.parquet`, `tracks_raw.parquet`, `build_log.json`.

Bật Internet để clone repo. Chỉ cần CPU. Chạy xong thì **Save Version**, rồi ở tab Output chọn **New Dataset**, đặt tên `mixnext-processed` (v1).

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #5 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "a"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
from pathlib import Path
INPUT = Path("/kaggle/input")
MPD_DIR = sorted(INPUT.rglob("mpd.slice.*.json"))[0].parent
FEATURES = sorted(INPUT.rglob("extracted.sqlite"))[0]
OUT = Path("/kaggle/working/processed")
print(MPD_DIR, FEATURES, sep="  |  ")

In [ ]:
import resource, time
from src.data.build import BuildConfig, build, write_build, format_log

CONFIG = BuildConfig()   # 100 slice, random_seed 42, bài ≥ 5 lượt, phiên ≥ 10 bài
t0 = time.time()
result = build(MPD_DIR, FEATURES, CONFIG)
paths = write_build(result, OUT)
print(format_log(result.log))
print(f"{time.time() - t0:.0f} giây · RAM đỉnh {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")
for p in paths:
    print(p, f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Chạy lại với cùng random_seed phải ra đúng cùng một tập (tiêu chí của #5).
# Chỉ giữ dấu vân tay của lần đầu để không phải giữ hai bản dữ liệu trong RAM.
import pandas as pd
def fingerprint(r):
    return (int(pd.util.hash_pandas_object(r.sessions, index=False).sum()), int(pd.util.hash_pandas_object(r.tracks, index=False).sum()))
first = fingerprint(result)
del result
print("giống hệt lần đầu:", fingerprint(build(MPD_DIR, FEATURES, CONFIG)) == first)

## Kết quả (chạy trên Kaggle, CPU, 24/11/2026)

`BuildConfig()`: 100 slice, `random_seed` = 42, bài ≥ 5 lượt, phiên ≥ 10 bài.

| bước | phiên | bài | lượt | % lượt |
|---|---:|---:|---:|---:|
| lấy mẫu 100 slice | 100.000 | 686.633 | 6.624.113 | 100,0% |
| bỏ bài không có features dùng được | 100.000 | 685.881 | 6.622.311 | 100,0% |
| vòng 1: bỏ bài < 5 lượt | 99.559 | 133.298 | 5.791.273 | 87,4% |
| vòng 1: bỏ phiên < 10 bài | 92.577 | 133.293 | 5.748.188 | 86,8% |
| vòng 2 | 92.430 | 130.921 | 5.738.123 | 86,6% |
| vòng 3 | 92.409 | 130.762 | 5.737.325 | 86,6% |
| vòng 4 | 92.406 | 130.739 | 5.737.212 | 86,6% |
| vòng 5–6 (ổn định) | **92.406** | **130.737** | **5.737.204** | **86,6%** |

- 876 giây, RAM đỉnh 1,9 GB. Lần chạy lại để kiểm tra mất khoảng 125 giây (đĩa đã có cache).
- `sessions_raw.parquet` 120,6 MB, `tracks_raw.parquet` 6,4 MB.
- Chạy lại với cùng `random_seed`: giống hệt lần đầu.